# tests.ipynb — the single source of truth

This notebook is scored most carefully (per the take-home brief).
It must:

1. Set up **both** baseline and candidate versions of scikit-image
   in the same Colab session (two separate installs, swapped via
   `pip install -e .` on a uninstall/reinstall cycle), so timing
   comparisons happen on the same VM.
2. Run the existing `test_regionprops.py` test suite on the
   candidate. Any test that passed at baseline but fails on the
   candidate is a regression → `existing_tests_pass = false`.
3. Compare candidate output against the baseline's
   `golden_output.npz` column-by-column with `np.allclose`,
   tolerance documented and justified.
4. Time both implementations with the **same** workload, **same**
   warmup count, **same** measured-run count.
5. Compute `speedup = baseline_median / candidate_median`.
6. Write `reward.json` with the exact schema from the brief.


## 1. Runtime info

In [ ]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version


## 1.5 Config — clone this submission repo

In [ ]:
# ====================================================================
# CONFIG (only thing the evaluator may need to change)
# --------------------------------------------------------------------
# Set the URL of THIS submission's GitHub repo (the one containing
# baseline_colab.ipynb, candidate_colab.ipynb, tests.ipynb, etc.).
# If you set the env var SUBMISSION_REPO_URL before opening Colab,
# this falls through to that value.
# ====================================================================
import os
SUBMISSION_REPO_URL = os.environ.get(
    "SUBMISSION_REPO_URL",
    "https://github.com/OWNER/REPO.git"   # <-- EDIT THIS if not set via env
)
SUBMISSION_DIR = "/content/submission"

if "OWNER/REPO" in SUBMISSION_REPO_URL:
    print("=" * 70)
    print("WARNING: SUBMISSION_REPO_URL is the placeholder.")
    print("Edit this cell to point at your fork, OR set:")
    print("  os.environ['SUBMISSION_REPO_URL'] = "
          "'https://github.com/yourname/yourrepo.git'")
    print("before re-running this cell.")
    print("=" * 70)
    raise SystemExit(0)

if not os.path.isdir(SUBMISSION_DIR):
    !git clone --depth 1 $SUBMISSION_REPO_URL $SUBMISSION_DIR

import sys
for p in (SUBMISSION_DIR,
          os.path.join(SUBMISSION_DIR, "workloads"),
          os.path.join(SUBMISSION_DIR, "benchmark"),
          os.path.join(SUBMISSION_DIR, "patches")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("Submission repo at:", SUBMISSION_DIR)


## 2. Common setup — both repo clones + submission repo

In [ ]:
import os, sys
WORK = "/content/work"
BASELINE_DIR  = os.path.join(WORK, "scikit-image-baseline")
CANDIDATE_DIR = os.path.join(WORK, "scikit-image-candidate")
os.makedirs(WORK, exist_ok=True)
for d in (BASELINE_DIR, CANDIDATE_DIR):
    if not os.path.isdir(d):
        !git clone --depth 1 --branch v0.18.3 https://github.com/scikit-image/scikit-image.git {d}
BASELINE_SHA = !git -C {BASELINE_DIR} rev-parse HEAD
BASELINE_SHA = BASELINE_SHA[0].strip()
print("baseline SHA:", BASELINE_SHA)


In [ ]:
# ---- Environment pinning (avoids dependency drift in Colab) -------------
# scikit-image v0.18.3 needs numpy < 1.24 and a compatible scipy.
!pip install -q --upgrade pip setuptools wheel
!pip install -q "numpy==1.23.5" "scipy==1.10.1" "cython<3" pytest line_profiler
!pip install -q networkx imageio tifffile pillow PyWavelets

import importlib, sys
for mod in ("numpy", "scipy"):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
import numpy as np, scipy
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)


## 3. Install BASELINE first, build workload, run baseline tests + timing

We install each version in turn, run its work, save outputs to
disk, then swap to the other version. This guarantees both
measurements happen on the same VM.


In [ ]:
def _import_skimage_fresh():
    import sys, importlib
    for k in [k for k in sys.modules if k.startswith('skimage')]:
        del sys.modules[k]
    import skimage
    return skimage

# ----- BASELINE install -----
!pip uninstall -y scikit-image > /dev/null 2>&1
%cd $BASELINE_DIR
!pip install -q -e . 2>&1 | tail -3
%cd /content
skimage = _import_skimage_fresh()
print("BASELINE skimage version:", skimage.__version__)
assert skimage.__version__ == "0.18.3"


In [ ]:
# Build workload (same as baseline_colab.ipynb).
from generate_workload import build_workload, WORKLOAD_PROPERTIES
import time, numpy as np
t0 = time.perf_counter()
labels, intensity, n_regions = build_workload()
print(f"workload: shape={labels.shape}, n_regions={n_regions} "
      f"({time.perf_counter()-t0:.2f}s)")


In [ ]:
# Run baseline tests, save pass set.
import subprocess, json
proc = subprocess.run(
    ["pytest", os.path.join(BASELINE_DIR,
     "skimage/measure/tests/test_regionprops.py"),
     "-v", "--tb=no", "-q", "--no-header"],
    capture_output=True, text=True,
)
baseline_passed, baseline_failed = [], []
for line in proc.stdout.splitlines():
    if " PASSED" in line:
        baseline_passed.append(line.split(" PASSED")[0].strip())
    elif " FAILED" in line:
        baseline_failed.append(line.split(" FAILED")[0].strip())
print(f"baseline: {len(baseline_passed)} pass, {len(baseline_failed)} fail")
os.makedirs("/content/outputs", exist_ok=True)
with open("/content/outputs/baseline_tests.json", "w") as f:
    json.dump({"passed": sorted(baseline_passed),
               "failed": sorted(baseline_failed)}, f, indent=2)


In [ ]:
# Save baseline output (golden) + measure baseline timing.
from skimage.measure import regionprops_table
golden = regionprops_table(labels, intensity_image=intensity,
                           properties=list(WORKLOAD_PROPERTIES))
np.savez_compressed("/content/outputs/golden_output.npz", **golden)

from benchmark_utils import bench
baseline_stats = bench(
    lambda: regionprops_table(labels, intensity_image=intensity,
                              properties=list(WORKLOAD_PROPERTIES)),
    label="baseline", n_warmup=2, n_measured=7
)


## 4. Switch to CANDIDATE — uninstall, reinstall, apply patch

In [ ]:
!pip uninstall -y scikit-image > /dev/null 2>&1
%cd $CANDIDATE_DIR
!pip install -q -e . 2>&1 | tail -3
%cd /content
# Apply our fast-path patch to this fresh install.
!python $SUBMISSION_DIR/patches/apply_optimization.py
skimage = _import_skimage_fresh()
print("CANDIDATE skimage version:", skimage.__version__)
from skimage.measure import _regionprops_fast    # raises if patch failed


In [ ]:
# Run the SAME test module against the candidate.
from skimage.measure import regionprops_table
proc = subprocess.run(
    ["pytest", os.path.join(CANDIDATE_DIR,
     "skimage/measure/tests/test_regionprops.py"),
     "-v", "--tb=no", "-q", "--no-header"],
    capture_output=True, text=True,
)
cand_passed, cand_failed = [], []
for line in proc.stdout.splitlines():
    if " PASSED" in line:
        cand_passed.append(line.split(" PASSED")[0].strip())
    elif " FAILED" in line:
        cand_failed.append(line.split(" FAILED")[0].strip())
print(f"candidate: {len(cand_passed)} pass, {len(cand_failed)} fail")
# Regression = test passed at baseline but fails on candidate.
regressions = sorted(set(baseline_passed) - set(cand_passed))
print(f"regressions: {len(regressions)}")
for r in regressions[:10]:
    print(" ", r)
existing_tests_pass = len(regressions) == 0


In [ ]:
# Save candidate output + measure candidate timing.
candidate_out = regionprops_table(labels, intensity_image=intensity,
                                  properties=list(WORKLOAD_PROPERTIES))
np.savez_compressed("/content/outputs/candidate_output.npz",
                    **candidate_out)

candidate_stats = bench(
    lambda: regionprops_table(labels, intensity_image=intensity,
                              properties=list(WORKLOAD_PROPERTIES)),
    label="candidate", n_warmup=2, n_measured=7
)


## 5. Output equivalence (golden vs candidate)

Tolerance: **1e-6 relative**. Justification: the fast path uses
`np.bincount` to accumulate row/col/intensity sums across the
whole image, where the baseline accumulates per region with
`np.sum`. The two are mathematically equal but accumulate in
different orders, so floating-point round-off can differ at
~1e-12 relative. Integer columns (label, area, bbox, bbox_area)
must match exactly.


In [ ]:
from benchmark_utils import compare_tables
ok, details = compare_tables(dict(golden), dict(candidate_out),
                             rtol=1e-6, atol=1e-6)
for k, v in details["columns"].items():
    print(f"  {k:30s}  match={v['match']}  "
          f"max_abs_diff={v.get('max_abs_diff', 'n/a')}")
output_equivalent = ok
print(f"\noutputs equivalent within 1e-6: {output_equivalent}")


## 6. Compute speedup + write reward.json

In [ ]:
from benchmark_utils import collect_environment, write_json

speedup = (baseline_stats["median"] / candidate_stats["median"]
           if existing_tests_pass and output_equivalent
           else None)

reward = {
    "repo": "scikit-image/scikit-image",
    "baseline_sha": BASELINE_SHA,
    "candidate_description": (
        "Vectorized regionprops_table fast path: scipy.ndimage.find_objects "
        "for all bboxes, np.bincount for areas/centroid sums/weighted centroid "
        "sums, scipy.ndimage.minimum/maximum for per-label intensity extrema; "
        "replaces the per-region Python dispatch loop for the common scalar "
        "property set, with safe fallback to the original implementation for "
        "unsupported properties."
    ),
    "baseline_time_s":  {k: baseline_stats[k]
                          for k in ("median", "iqr", "n_warmup", "n_measured")},
    "candidate_time_s": {k: candidate_stats[k]
                          for k in ("median", "iqr", "n_warmup", "n_measured")},
    "speedup": (float(speedup) if speedup is not None else None),
    "correctness": {
        "existing_tests_pass": bool(existing_tests_pass),
        "output_equivalent":   bool(output_equivalent),
        "tolerance": "1e-6 relative",
        "tolerance_justification": (
            "np.bincount accumulates sums across the full image; the baseline "
            "uses per-region np.sum. Both compute identical mathematical "
            "quantities but in different orders, so float results differ at "
            "~1e-12 relative. Integer columns (label, area, bbox, bbox_area) "
            "are required to match exactly."
        ),
    },
    "environment": collect_environment(),
}
write_json("/content/reward.json", reward)
import json
print(json.dumps(reward, indent=2))


In [ ]:
# One more sanity print so an evaluator can eyeball the result.
print("=" * 60)
print(f"BASELINE  median = {baseline_stats['median']:.4f}s  "
      f"IQR = {baseline_stats['iqr']:.4f}s")
print(f"CANDIDATE median = {candidate_stats['median']:.4f}s  "
      f"IQR = {candidate_stats['iqr']:.4f}s")
if reward["speedup"] is not None:
    print(f"SPEEDUP = {reward['speedup']:.2f}x")
else:
    print("SPEEDUP = null  (correctness failed; speedup not reported)")
print(f"existing_tests_pass = {reward['correctness']['existing_tests_pass']}")
print(f"output_equivalent   = {reward['correctness']['output_equivalent']}")
